# Bronze Layer — Ingestie SQLite → Delta Lake

**Scop:** Citirea datelor brute din `banking.db` (SQLite) și încărcarea lor în tabele Delta din `banking.bronze`, fără transformări.

**Principii Bronze:**
- Datele sunt copiate **1:1** din sursă, fără curățare sau validare
- Se adaugă **metadata coloane** pentru trasabilitate: `_ingestion_ts`, `_source_system`, `_batch_id`, `_source_file`
- Erorile injectate (date murdare) trec **nemodificate** — vor fi tratate în Silver
- Format: **Delta Lake** (suportă ACID, time travel, schema evolution)

**Strategie de încărcare:** `overwrite` la fiecare rulare (snapshot complet din SQLite). Pentru un sistem real cu volume mari, ai folosi CDC (Change Data Capture) cu `merge`, dar pentru proiectul nostru `overwrite` e suficient și mai ușor de demonstrat.

## 1. Configurare și constante

In [0]:
import sqlite3
import uuid
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

# Calea catre SQLite uploadat in Volume
SQLITE_PATH = "/Volumes/banking/bronze/raw_files/banking.db"

# Catalog si schema destinatie
CATALOG = "banking"
BRONZE_SCHEMA = "bronze"

# Identificator unic pentru aceasta rulare (pentru trasabilitate)
BATCH_ID = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
SOURCE_SYSTEM = "sqlite_banking"

print(f"Batch ID:       {BATCH_ID}")
print(f"Source system:  {SOURCE_SYSTEM}")
print(f"SQLite path:    {SQLITE_PATH}")
print(f"Destinatie:     {CATALOG}.{BRONZE_SCHEMA}.*")

Batch ID:       batch_20260501_193857_66b4f503
Source system:  sqlite_banking
SQLite path:    /Volumes/banking/bronze/raw_files/banking.db
Destinatie:     banking.bronze.*


## 2. Verificare fișier sursă

Ne asigurăm că `banking.db` există în Volume și este accesibil.

In [0]:
# COMMAND ----------

import os
import shutil

# Calea sursa in Volume (read-only pentru SQLite din motive I/O)
SQLITE_VOLUME_PATH = "/Volumes/banking/bronze/raw_files/banking.db"

# Calea locala pe driver (suporta operatii SQLite complete)
SQLITE_LOCAL_PATH = "/tmp/banking_local.db"

if not os.path.exists(SQLITE_VOLUME_PATH):
    raise FileNotFoundError(
        f"Fisierul SQLite nu exista la {SQLITE_VOLUME_PATH}. "
        f"Verifica ca ai uploadat banking.db in Volume."
    )

# Copiem fisierul local — SQLite are nevoie sa scrie WAL/journal
print(f"Copiere {SQLITE_VOLUME_PATH} -> {SQLITE_LOCAL_PATH} ...")
shutil.copy(SQLITE_VOLUME_PATH, SQLITE_LOCAL_PATH)

file_size_mb = os.path.getsize(SQLITE_LOCAL_PATH) / (1024 * 1024)
print(f"banking.db gata pentru citire | Dimensiune: {file_size_mb:.2f} MB")

# IMPORTANT: de aici incolo folosim SQLITE_LOCAL_PATH, nu SQLITE_VOLUME_PATH
SQLITE_PATH = SQLITE_LOCAL_PATH

Copiere /Volumes/banking/bronze/raw_files/banking.db -> /tmp/banking_local.db ...
banking.db gata pentru citire | Dimensiune: 33.06 MB


## 3. Descoperire tabele din SQLite

Ne conectăm la baza SQLite și listăm toate tabelele disponibile pentru ingestie.

In [0]:
def get_sqlite_tables(db_path: str) -> list[dict]:
    """
    Returneaza lista tuturor tabelelor utilizator din SQLite,
    cu numarul de randuri si coloanele lor.
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    
    # Toate tabelele user-defined (excludem tabele interne sqlite_*)
    tables_info = conn.execute("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()
    
    result = []
    for t in tables_info:
        table_name = t["name"]
        count = conn.execute(f"SELECT COUNT(*) FROM [{table_name}]").fetchone()[0]
        cols = [c["name"] for c in conn.execute(f"PRAGMA table_info([{table_name}])").fetchall()]
        result.append({
            "table"  : table_name,
            "rows"   : count,
            "columns": cols,
        })
    
    conn.close()
    return result


tables = get_sqlite_tables(SQLITE_PATH)
print(f"Gasite {len(tables)} tabele in SQLite:\n")
for t in tables:
    print(f"  {t['table']:35s} {t['rows']:>7} randuri | {len(t['columns'])} coloane")

Gasite 22 tabele in SQLite:

  accounts                                800 randuri | 13 coloane
  audit_log                             50252 randuri | 11 coloane
  branches                                 20 randuri | 13 coloane
  cards                                   600 randuri | 12 coloane
  customers                               500 randuri | 18 coloane
  employees                                60 randuri | 12 coloane
  loans                                   200 randuri | 16 coloane
  ref_account_types                         4 randuri | 7 coloane
  ref_card_types                            6 randuri | 7 coloane
  ref_channels                              7 randuri | 7 coloane
  ref_counties                             42 randuri | 5 coloane
  ref_countries                            20 randuri | 8 coloane
  ref_currencies                            6 randuri | 8 coloane
  ref_customer_segments                     4 randuri | 6 coloane
  ref_employee_roles                    

## 4. Funcție generică de ingestie

Pentru fiecare tabel SQLite:
1. Citim cu `pandas.read_sql` (rapid pentru volume mici-medii)
2. Convertim la Spark DataFrame
3. Adăugăm coloanele de metadata
4. Scriem ca tabel Delta cu `mode("overwrite")` și `mergeSchema=true` (pentru schema evolution)

**Note tehnice:**
- `mergeSchema=true` permite ca dacă în SQLite apar coloane noi, schema Delta să se adapteze automat
- `overwriteSchema=true` permite recrearea completă a schemei dacă tipurile s-au schimbat
- Folosim `option("delta.columnMapping.mode", "name")` pentru a permite caractere speciale în nume

In [0]:
def ingest_table_to_bronze(
    sqlite_path: str,
    table_name: str,
    catalog: str,
    schema: str,
    batch_id: str,
    source_system: str,
) -> dict:
    """
    Citeste o tabela din SQLite si o scrie ca tabel Delta in Bronze.
    Returneaza un dict cu statistici despre operatie.
    """
    start_time = datetime.now()
    
    # 1. Citim din SQLite cu pandas (mai rapid decat row-by-row)
    conn = sqlite3.connect(sqlite_path)
    pdf = pd.read_sql_query(f"SELECT * FROM [{table_name}]", conn)
    conn.close()
    
    rows_source = len(pdf)
    
    # Cazul edge: tabel gol
    if rows_source == 0:
        # Cream o tabela goala cu schema corecta + metadata
        # Folosim toate coloanele ca STRING (suficient pentru Bronze)
        col_names = pdf.columns.tolist() if not pdf.empty else []
        if not col_names:
            # Cerem schema direct din SQLite
            conn = sqlite3.connect(sqlite_path)
            cols = conn.execute(f"PRAGMA table_info([{table_name}])").fetchall()
            conn.close()
            col_names = [c[1] for c in cols]
        
        empty_schema = StructType([StructField(c, StringType(), True) for c in col_names])
        sdf = spark.createDataFrame([], empty_schema)
    else:
        # Convertim toate coloanele la string pentru Bronze
        # (Bronze nu face type casting — pastram ca text, Silver va converti)
        pdf = pdf.astype(str).where(pdf.notna(), None)
        sdf = spark.createDataFrame(pdf)
    
    # 2. Adaugam coloane de metadata
    sdf_with_meta = (
        sdf
        .withColumn("_ingestion_ts",  current_timestamp())
        .withColumn("_source_system", lit(source_system))
        .withColumn("_batch_id",      lit(batch_id))
        .withColumn("_source_file",   lit("banking.db"))
    )
    
    # 3. Scriem in Delta — overwrite la fiecare rulare
    target_table = f"{catalog}.{schema}.raw_{table_name}"
    
    (sdf_with_meta.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table))
    
    elapsed = (datetime.now() - start_time).total_seconds()
    
    return {
        "source_table"   : table_name,
        "target_table"   : target_table,
        "rows"           : rows_source,
        "elapsed_seconds": round(elapsed, 2),
    }

## 5. Rulare ingestie pentru toate tabelele

Iterăm prin lista de tabele descoperite și încărcăm fiecare în Bronze.

In [0]:
results = []
total_rows = 0
total_start = datetime.now()

print(f"START ingestie Bronze — Batch: {BATCH_ID}\n")
print(f"{'Tabel sursa':35s} {'Randuri':>10s} {'Timp (s)':>10s}  Tabel destinatie")
print("-" * 100)

for t in tables:
    try:
        result = ingest_table_to_bronze(
            sqlite_path   = SQLITE_PATH,
            table_name    = t["table"],
            catalog       = CATALOG,
            schema        = BRONZE_SCHEMA,
            batch_id      = BATCH_ID,
            source_system = SOURCE_SYSTEM,
        )
        results.append(result)
        total_rows += result["rows"]
        print(
            f"{result['source_table']:35s} "
            f"{result['rows']:>10,d} "
            f"{result['elapsed_seconds']:>10.2f}  "
            f"{result['target_table']}"
        )
    except Exception as e:
        print(f"  EROARE la {t['table']}: {e}")
        results.append({"source_table": t["table"], "error": str(e)})

total_elapsed = (datetime.now() - total_start).total_seconds()

print("-" * 100)
print(f"TOTAL: {len(results)} tabele | {total_rows:,} randuri | {total_elapsed:.1f}s")

START ingestie Bronze — Batch: batch_20260501_193857_66b4f503

Tabel sursa                            Randuri   Timp (s)  Tabel destinatie
----------------------------------------------------------------------------------------------------
accounts                                   800       2.77  banking.bronze.raw_accounts
audit_log                               50,252       4.01  banking.bronze.raw_audit_log
branches                                    20       1.97  banking.bronze.raw_branches
cards                                      600       1.96  banking.bronze.raw_cards
customers                                  500       2.24  banking.bronze.raw_customers
employees                                   60       1.96  banking.bronze.raw_employees
loans                                      200       1.98  banking.bronze.raw_loans
ref_account_types                            4       2.58  banking.bronze.raw_ref_account_types
ref_card_types                               6       2.34 

## 6. Verificare tabele Bronze create

Listăm toate tabelele din `banking.bronze` ca să confirmăm că ingestia a reușit.

In [0]:
%sql
SHOW TABLES IN banking.bronze;

database,tableName,isTemporary
bronze,_bronze_runs,false
bronze,raw_accounts,false
bronze,raw_audit_log,false
bronze,raw_branches,false
bronze,raw_cards,false
bronze,raw_customers,false
bronze,raw_employees,false
bronze,raw_loans,false
bronze,raw_ref_account_types,false
bronze,raw_ref_card_types,false


## 7. Sumar Bronze — Row counts și metadata

Verificăm că fiecare tabel Bronze conține randurile așteptate și metadata corectă.

In [0]:
%sql
-- Numarul total de randuri per tabel + batch_id curent
SELECT 
    'raw_branches'             AS tabel, COUNT(*) AS randuri, MAX(_batch_id) AS ultim_batch FROM banking.bronze.raw_branches
UNION ALL SELECT 'raw_employees',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_employees
UNION ALL SELECT 'raw_customers',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_customers
UNION ALL SELECT 'raw_accounts',           COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_accounts
UNION ALL SELECT 'raw_cards',              COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_cards
UNION ALL SELECT 'raw_loans',              COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_loans
UNION ALL SELECT 'raw_transactions',       COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_transactions
UNION ALL SELECT 'raw_audit_log',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_audit_log
UNION ALL SELECT 'raw_ref_countries',      COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_countries
UNION ALL SELECT 'raw_ref_counties',       COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_counties
UNION ALL SELECT 'raw_ref_currencies',     COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_currencies
UNION ALL SELECT 'raw_ref_transaction_types',  COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_transaction_types
UNION ALL SELECT 'raw_ref_transaction_statuses',COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_transaction_statuses
UNION ALL SELECT 'raw_ref_customer_segments',   COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_customer_segments
UNION ALL SELECT 'raw_ref_account_types',       COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_account_types
UNION ALL SELECT 'raw_ref_loan_types',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_loan_types
UNION ALL SELECT 'raw_ref_card_types',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_card_types
UNION ALL SELECT 'raw_ref_channels',            COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_channels
UNION ALL SELECT 'raw_ref_risk_bands',          COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_risk_bands
UNION ALL SELECT 'raw_ref_kyc_statuses',        COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_kyc_statuses
UNION ALL SELECT 'raw_ref_employee_roles',      COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_employee_roles
UNION ALL SELECT 'raw_ref_regions',             COUNT(*), MAX(_batch_id) FROM banking.bronze.raw_ref_regions
ORDER BY tabel;

tabel,randuri,ultim_batch
raw_accounts,800,batch_20260501_193857_66b4f503
raw_audit_log,50252,batch_20260501_193857_66b4f503
raw_branches,20,batch_20260501_193857_66b4f503
raw_cards,600,batch_20260501_193857_66b4f503
raw_customers,500,batch_20260501_193857_66b4f503
raw_employees,60,batch_20260501_193857_66b4f503
raw_loans,200,batch_20260501_193857_66b4f503
raw_ref_account_types,4,batch_20260501_193857_66b4f503
raw_ref_card_types,6,batch_20260501_193857_66b4f503
raw_ref_channels,7,batch_20260501_193857_66b4f503


## 8. Inspecție rapidă — exemplu `raw_transactions`

Verificăm că datele "murdare" (erori injectate) au ajuns în Bronze. Acestea vor fi prinse de Silver.

In [0]:
%sql
-- Exemple de tranzactii murdare (vor merge in carantina la Silver)
SELECT 
    transaction_id,
    status_code,
    channel_code,
    amount,
    currency_code,
    _ingestion_ts,
    _batch_id
FROM banking.bronze.raw_transactions
WHERE 
    -- Erori tehnice: status invalid, channel invalid, amount NULL, amount negativ
    status_code  = 'INVALID_STATUS'
    OR channel_code = 'INVALID_CHANNEL'
    OR amount IS NULL
    OR amount = 'None'
    OR CAST(amount AS DOUBLE) < 0
LIMIT 20;

transaction_id,status_code,channel_code,amount,currency_code,_ingestion_ts,_batch_id
TXN-00044656,COMPLETED,INVALID_CHANNEL,19.02,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044697,PENDING,INVALID_CHANNEL,57.14,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044757,INVALID_STATUS,POS,97.42,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044778,INVALID_STATUS,ATM,469.63,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044784,COMPLETED,INVALID_CHANNEL,23.56,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044822,FAILED,INVALID_CHANNEL,62.36,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044833,COMPLETED,INVALID_CHANNEL,512.6,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044880,COMPLETED,INVALID_CHANNEL,726.0,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044912,INVALID_STATUS,POS,462.42,EUR,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503
TXN-00044919,INVALID_STATUS,ATM,1057.45,RON,2026-05-01T19:39:49.954Z,batch_20260501_193857_66b4f503


## 9. Time Travel Demo — capabilitate Delta Lake

Una dintre cele mai puternice features Delta Lake este `time travel` — putem accesa orice versiune istorică a unui tabel.

In [0]:
%sql
-- Vedem istoricul versiunilor pentru raw_transactions
DESCRIBE HISTORY banking.bronze.raw_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-05-01T19:39:52.000Z,77188429615603,rusuiulian21@stud.ase.ro,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3800153432022618),b543858e-4333-4014-bd32-c50d0b325467,0501-181445-k8t8spn4-v2n,1,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 1923623, numDeletionVectorsRemoved -> 0, numOutputRows -> 50252, numOutputBytes -> 1921136)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-30T14:51:32.000Z,77188429615603,rusuiulian21@stud.ase.ro,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3800153432022618),1dc9cdea-f4d2-45d5-a888-9ba56c8f2324,0430-142645-or7q04gs-v2n,0,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 1922387, numDeletionVectorsRemoved -> 0, numOutputRows -> 50352, numOutputBytes -> 1923623)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-30T14:29:31.000Z,77188429615603,rusuiulian21@stud.ase.ro,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3800153432022618),a99fbbae-3450-4f96-b8fd-a6c58f9dd59a,0430-142645-or7q04gs-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 50302, numOutputBytes -> 1922387)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


## 10. Logging batch run — pentru audit

Salvăm metadata rulării într-o tabelă specială `_bronze_runs` pentru a putea urmări istoricul rulărilor pipeline-ului.

In [0]:
import json
from pyspark.sql import Row

# Construim un row cu metadata rularii
run_metadata = Row(
    batch_id        = BATCH_ID,
    started_at      = total_start.isoformat(),
    completed_at    = datetime.now().isoformat(),
    elapsed_seconds = float(total_elapsed),
    tables_count    = len(results),
    total_rows      = int(total_rows),
    source_system   = SOURCE_SYSTEM,
    source_file     = SQLITE_PATH,
    details_json    = json.dumps(results),
)

run_df = spark.createDataFrame([run_metadata])

# Append in tabela de runs (creata daca nu exista)
(run_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}._bronze_runs"))

print(f"Run inregistrat in {CATALOG}.{BRONZE_SCHEMA}._bronze_runs")
print(f"Batch ID: {BATCH_ID}")

Run inregistrat in banking.bronze._bronze_runs
Batch ID: batch_20260501_193857_66b4f503


In [0]:
%sql
-- Verificare: ultimele 5 rulari ale pipeline-ului Bronze
SELECT 
    batch_id, 
    started_at, 
    elapsed_seconds, 
    tables_count, 
    total_rows
FROM banking.bronze._bronze_runs
ORDER BY started_at DESC
LIMIT 5;

batch_id,started_at,elapsed_seconds,tables_count,total_rows
batch_20260501_193857_66b4f503,2026-05-01T19:38:59.984411,52.03121,22,102818
batch_20260430_145046_d0be7dbf,2026-04-30T14:50:48.398058,43.812628,22,103293


## Sumar Bronze

Ce am realizat:
- Citit `banking.db` din Volume (`/Volumes/banking/bronze/raw_files/`)
- Descoperit dinamic toate cele 22 tabele din SQLite
- Încărcat fiecare tabelă ca Delta în `banking.bronze.raw_*`
- Adăugat metadata pentru trasabilitate: `_ingestion_ts`, `_source_system`, `_batch_id`, `_source_file`
- Păstrat datele "murdare" intacte (erori injectate vor fi tratate în Silver)
- Înregistrat metadata rulării în `_bronze_runs` pentru audit

**Următorul pas:** Notebook `02_silver_dlt` — pipeline Delta Live Tables care:
- Aplică validări pe Bronze
- Separă datele valide de cele murdare (quarantine)
- Construiește modelul dimensional (dim_*, fact_*)
- Implementează SCD Type 2 pe `dim_customers`